# 27_03 랜덤 포레스트 고장 임박 분류


In [9]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

✅ 환경 설정 완료! 현재 적용된 폰트: ['AppleGothic']


## 모델 만들고 학습시키기
모듈 2에서 만든 X_train, y_train으로 랜덤 포레스트 학습

In [10]:
# ==============================================================================
# 1. 데이터 로드 및 타겟 변수(y) 생성
# ==============================================================================
# C-MAPSS FD001 데이터셋 불러오기

df = pd.read_csv("../csv/27_cmapss_fd001_sample.csv", encoding="utf-8")

# 타겟 변수 생성: 잔여수명(RUL)이 30 이하이면 고장 임박(1), 아니면 정상(0)
df["failure_soon"] = (df["RUL"] <= 30).astype(int)

# 타겟 클래스 불균형 확인 (고장 임박 vs 정상 데이터 개수 및 비율)
print("--- [클래스 개수 확인] ---")
print(df["failure_soon"].value_counts()) 

print("\n--- [클래스 비율 확인] ---")
print(df["failure_soon"].value_counts(normalize=True)) 


# ==============================================================================
# 2. 특성(X)과 타겟(y) 정의
# ==============================================================================
# 예측 모델 학습에 사용할 센서 피처 선택
feats = ["sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_11", "sensor_15"]

X = df[feats]            # 입력 특성 (Features)
y = df["failure_soon"]   # 예측할 정답 (Target)


# ==============================================================================
# 3. 데이터셋 분할 (Train / Test Split)
# ==============================================================================

# scikit-learn에서 데이터를 나눠 주는 도구 불러오기
from sklearn.model_selection import train_test_split

# [분할 목적]
# - 학습용 데이터와 평가용 데이터를 엄격히 Seprate하여 오버피팅(암기)을 방지함
# - 모델이 학습 데이터만 외우지 않고, 새로운 데이터에 대해 일반화(응용)할 수 있는지 검증
#
# [파라미터 설명]
# - test_size = 0.2  : 전체 데이터 중 20%를 테스트용으로 분할
# - random_state = 42: 난수 고정으로 코드 실행 시마다 동일한 분할 결과 유지
# - stratify = y     : y(타겟)의 0과 1 비율을 Train과 Test에 동일하게 유지 (불균형 방지)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# ==============================================================================
# 4. 분할 결과 검증 및 출력
# ==============================================================================
print("\n==================== [데이터 차원(Shape) 확인] ====================")
print("X_train shape : ", X_train.shape)  # 학습용 피처 (행, 열)
print("X_test  shape : ", X_test.shape)   # 평가용 피처 (행, 열)
print("y_train shape : ", y_train.shape)  # 학습용 정답 (행,)
print("y_test  shape : ", y_test.shape)   # 평가용 정답 (행,)

print("\n==================== [샘플 데이터 확인 (상위 3개)] ====================")
print("==== X_train ====\n", X_train.head(3), "\n")
print("==== X_test  ====\n", X_test.head(3), "\n")
print("==== y_train ====\n", y_train.head(3), "\n")
print("==== y_test  ====\n", y_test.head(3))

ValueError: binary mode doesn't take an encoding argument

### 분류기 불러오기
`sklearn.ensemble`에서 `RandomForestClassifier` 임포트

In [ ]:
# 코드
# scikit-learn(sklearn)의 ensemble(앙상블) 모듈에서 RandomForestClassifier(랜덤 포레스트 분류기) 클래스를 불러옵니다.
# (여러 개의 의사결정나무를 조합해 정답(범주)을 예측하는 대표적인 머신러닝 모델)

from sklearn.ensemble import RandomForestClassifier

### 모델 생성
트리 100그루 모델 생성 — 아직 학습 전 빈 상자


In [ ]:
# 코드
# 1. 기본 설정: n_estimators 기본값(100) 적용, 실행할 때마다 결과가 조금씩 달라짐 (비복원/복원 추출 및 특성 선택의 무작위성 때문)
# model = RandomForestClassifier()

# 2. 명시적 트라이 설정: 나무 개수를 100개로 명확히 지정 (라이브러리 버전에 상관없이 동일한 개수 보장, 여전히 실행마다 결과 변동)
# model = RandomForestClassifier(n_estimators=100)

# 3. 재현성 확보: 나무 100개 지정 + 난수 시드 고정 (언제 실행해도 항상 100% 동일한 모델 결과 출력 -> 비교 및 실습에 필수)
model = RandomForestClassifier(n_estimators=100, random_state=42)

### 학습 실행
학습용 데이터로 학습 — 평가용은 절대 넣지 않음


In [ ]:
# 코드
# 파일 읽기 기본 인코딩을 utf-8로 지정
import builtins

# 파일 읽기 기본 인코딩을 utf-8로 지정
_orig_open = builtins.open
def _fixed_open(*args, **kwargs):
    kwargs.setdefault('encoding', 'utf-8')
    return _orig_open(*args, **kwargs)

builtins.open = _fixed_open
# .fit() # 모델에게 "데이터 패턴을 학습하라"고 지시하는 scikit-learn의 핵심 메서드

# 입력 데이터(X_train)와 정답 라벨(y_train)의 관계와 패턴을 머신러닝 모델이 공부(학습)하도록 명령합니다.
# 모델 학습 (문제와 정답을 함께 전달하여 패턴 습득)

model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

### 학습 확인
모델 변수를 출력해 학습이 끝났는지 확인


In [ ]:

#  모델 확인 (이제 아래에 원하시던 상세 표/다이어그램이 출력됩니다)
model

## 🌲 RandomForestClassifier 학습 결과 속성 요약
💡 참고

속성명 끝에 붙은 **언더스코어(_)**는 모델이 데이터 학습(fit)을 완료한 후 자동으로 생성된 결과임을 의미합니다.

## 1. 📊 모델 구조 및 입출력 정보
feature_names_in_
의미: 모델 학습 시 사용된 입력 변수(컬럼)들의 이름

현재 값: ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_15']

n_features_in_
의미: 모델에 입력된 특성(컬럼)의 총 개수

현재 값: 6 (6개의 센서 데이터 사용)

classes_
의미: 분류할 정답(클래스)의 종류

현재 값: [0, 1] (이진 분류)

n_classes_
의미: 분류할 클래스의 총 개수

현재 값: 2

n_outputs_
의미: 예측하고자 하는 타겟(목표 변수)의 개수

현재 값: 1 (단일 목표값 예측)

## 2. 🎯 특성 중요도 (Feature Importance)
feature_importances_
의미: 각 입력 변수가 예측 결과에 미치는 중요도(영향력) (0~1 사이 값, 전체 합 = 1)

현재 값: [0.16, 0.07, 0.12, 0.28, 0.15, 0.21]

분석:

sensor_7 (0.28): 예측에 가장 큰 영향을 미치는 핵심 변수

sensor_3 (0.07): 예측에 미치는 영향이 가장 적은 변수

## 3. 🌳 앙상블(숲) 내부 트리의 구성 정보
estimator_
의미: 랜덤 포레스트를 구성하는 기본 단위 모델(Base Estimator)

현재 값: DecisionTreeClassifer() (의사결정나무 사용)

estimators_
의미: 앙상블을 위해 생성된 개별 의사결정나무 객체들의 전체 리스트

현재 값: 수십~수백 개의 DecisionTree 리스트 (개별 트리를 선택해 visualizer 등으로 시각화 가능)

estimators_samples_
의미: 각 나무를 만들 때 복원 추출(Bootstrap)로 사용된 데이터 샘플의 인덱스 목록

현재 값: 각 나무가 학습할 때 사용한 데이터 번호 배열

💡 참고

속성명 끝에 붙은 **언더스코어(_)**는 모델이 데이터 학습(fit)을 완료한 후 자동으로 생성된 결과임을 의미합니다.

## 1. 📊 모델 구조 및 입출력 정보
feature_names_in_
의미: 모델 학습 시 사용된 입력 변수(컬럼)들의 이름

현재 값: ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_15']

n_features_in_
의미: 모델에 입력된 특성(컬럼)의 총 개수

현재 값: 6 (6개의 센서 데이터 사용)

classes_
의미: 분류할 정답(클래스)의 종류

현재 값: [0, 1] (이진 분류)

## 예측하고 결과 형태 확인
- 평가용 데이터로 예측 받고 결과가 0/1 배열임을 확인



### 예측 실행
예측 결과를 y_pred에 담고 출력 — 0/1로만 이루어진 배열인가?


In [ ]:
# 코드
y_pred = model.predict(X_test)
y_pred

### 개수 확인
예측 개수와 평가용 행 개수가 같은지 확인


In [ ]:
# 코드
print(len(y_pred))
print(X_test.shape)

### 앞부분 보기
예측의 앞 10개만 잘라 보기 — 순서는 X_test와 동일


In [ ]:
# 코드
y_pred[:10]

## 결과 형태 정리
- 예측 = 행마다 0/1이 담긴 배열, 입력과 같은 순서
## predict_proba로 확신도 살펴보기
- 각 행의 0일 확률과 1일 확률 확인 — 강한 확신/애매한 행 비교


### 확률 받기
예측 확률을 받아 앞 5개 확인 — 두 확률을 더하면 1이 되는가?


In [ ]:
# 코드

# [1] 모델 예측 확률 계산
# proba shape: (N_samples, N_classes) -> 예: (테스트데이터수, 2)
proba = model.predict_proba(X_test)

# [2] 상위 5개 샘플의 예측 확률 확인
# ----------------------------------------------------------------------------
# [확인하는 이유]
# 1. 모델 동작 검증: 예측 확률이 정상 범위([0, 1] 사이, 합=1)로 나오는지 확인
# 2. 클래스 순서 파악: [Class 0 확률, Class 1 확률] 순서 체크 (model.classes_와 일치)
# 3. 데이터 편향 체크: 한쪽 클래스로 지나치게 쏠려있는지 초기 점검
# 4. 후속 작업 준비: 양성 확률만 추출(proba[:, 1])하거나 임계값을 변경할지 결정
# ----------------------------------------------------------------------------
# 출력 형태: [[Class 0 확률, Class 1 확률], ...]

proba[:5]

# 현재 결과: 5개 샘플 모두 Class 0(음성/첫번째 클래스)일 확률이 88%~100%로 매우 높음
# array([[0.99, 0.01],  # 1번 샘플: Class 0 (99%), Class 1 (1%)
#        [1.  , 0.  ],  # 2번 샘플: Class 0 (100%), Class 1 (0%)
#        [0.88, 0.12],  # 3번 샘플: Class 0 (88%), Class 1 (12%)
#        [1.  , 0.  ],  # 4번 샘플: Class 0 (100%), Class 1 (0%)
#        [1.  , 0.  ]]) # 5번 샘플: Class 0 (100%), Class 1 (0%)

### 1일 확률만 보기
두 번째 열, 즉 1일 확률(곧 고장 확률)만 따로 보기


In [ ]:
# 코드
# 상위 5개 샘플의 Class 1(양성) 예측 확률만 추출 (shape: [5,])
# ----------------------------------------------------------------------------
# [확인 이유 및 목적]
# 1. 관심 대상(Class 1)에 대한 확률값만 1차원 배열로 간단히 확인
# 2. ROC-AUC, PR-AUC 등 확률 기반 평가 지표 함수에 입력하기 전 데이터 확인
# 3. 임계값(Threshold) 조정 시 기준이 되는 '양성 확률' 슬라이싱 동작 체크
# ----------------------------------------------------------------------------
proba[:5, 1]

# [출력 결과 해석]
# array([0.01, 0.  , 0.12, 0.  , 0.  ])
# -> 1번 샘플: 1% / 2번 샘플: 0% / 3번 샘플: 12% / 4번 샘플: 0% / 5번 샘플: 0%
# -> 상위 5개 샘플은 모두 Class 1(양성)일 확률이 매우 낮아 Class 0으로 분류됨

## 확신도 해석
- 확률이 높은 행과 0.5 부근인 행을 비교해 확신도 차이를 느껴 보기
## random_state 바꿔 다시 학습
- 값을 바꾸면 결과가 조금 달라짐, 같은 값이면 똑같이 재현됨


### 다른 random_state로 학습
random_state를 7로 바꿔 새 모델을 학습하고 예측


In [ ]:
# 코드

model2 = RandomForestClassifier(n_estimators=100, random_state=7)
# model2 # 확인

model2.fit(X_train, y_train)

y_pred2 = model2.predict(X_test)

### 예측 비교
앞 예측과 새 예측의 앞부분 비교 — 일부 예측이 조금 달라졌는가?

In [ ]:
# 코드
print(y_pred[:10])
print(y_pred2[:10])

### 같은 값이면 재현
같은 random_state 42로 다시 학습해 결과가 똑같은지 확인


In [ ]:
# 코드

# [1] RandomForest 모델 생성 (트리 100개, 시드 고정)
model3 = RandomForestClassifier(n_estimators=100, random_state=42)

# [2] 학습 데이터(X_train, y_train)로 모델 학습
model3.fit(X_train, y_train)

# [3] 테스트 데이터 전체의 예측 결과가 실제 정답과 '완벽히 일치'하는지 검증
# ----------------------------------------------------------------------------
# [이 코드를 작성한 이유 및 목적]
# 1. 단 하나의 오차도 없이 테스트 세트를 100% 맞췄는지(Accuracy = 1.0) 확인
# 2. 파이프라인이나 데이터에 오버피팅, 데이터 누수(Data Leakage), 
#    또는 완벽히 분리 가능한 쉬운 데이터셋인지 빠르게 단산(True/False) 판단
# 3. 테스트 세트 크기가 매우 작을 때 전수 적중 여부를 직관적으로 테스트
# ----------------------------------------------------------------------------
(model3.predict(X_test) == y_test).all()
# True  -> 테스트 데이터 전체의 예측이 100% 맞음 (정답률 100%)
# False -> 단 1개라도 틀린 예측 샘플이 존재함

## 예측 비교 + feature importance
- 예측·실제 눈 비교 + 어떤 센서가 판단에 크게 쓰였는지 확인


### 예측·실제 나란히 보기
예측과 실제를 한 표에 담아 앞부분 확인


In [ ]:
# 코드

# 예측 : y_pred
# 실제 : y_test.values
print("예측값 상위 15개:\n", y_pred[:15])
print("실제값 상위 15개:\n", y_test.values[:15])

compare = pd.DataFrame({"예측": y_pred, "실제": y_test.values})
compare.head()

## 눈으로 일치 세기
- 표를 보며 예측과 실제가 같은 행이 많은지 확인


### 중요 센서 확인
각 센서의 중요도를 큰 순으로 정렬


In [ ]:
# 코드

importance = pd.Series(model.feature_importances_, index = feats)

print(importance.sort_values(ascending=False))

## 중요도 활용
- 중요한 센서 부위를 우선 점검 안내 — 모델 판단이 현장 행동으로
## 예측 결과를 표로 모아 비교
- 예측·실제·곧고장확률을 한 표에 모아 정비 우선순위로 가공


### 결과 표 만들기
예측·실제·곧고장확률을 한 표에 — 확률은 1일 확률 사용


In [ ]:
# 코드

result = pd.DataFrame({'예측': y_pred, "실제": y_test.values, "곧고장확률": proba[:, 1]})
result.head(15)

### 곧 고장만 추리기
1로 예측된 행만 추리기 — 우선 점검 후보


In [ ]:
# 코드
result[result['예측'] == 1].head(10)

### 우선순위 정렬
곧 고장 확률 높은 순으로 정렬해 우선순위 만들기


In [ ]:
# 코드
result.sort_values('곧고장확률', ascending=False).head(10)

## 표 활용 정리
- 정렬된 표 자체가 정비 우선순위 목록 — 그대로 정비팀 전달 가능
## MIMII feature CSV로 정상/이상 분류
- 소리 특징 데이터에 똑같은 분류 흐름 적용 — 데이터만 바뀔 뿐


### 소리 데이터 불러오기
`rms`, `spectral_centroid`, `zero_crossing_rate`, `label` 컬럼 확인

In [ ]:
# 코드
mdf = pd.read_csv("27_mimii_features_sample.csv")
mdf.head()

### X/y 나누기
소리 특징 세 개를 입력 mX로, label을 정답 my로 분리


In [ ]:
# 코드
mfeats = ["rms",	"spectral_centroid",	"zero_crossing_rate"]

mX = mdf[mfeats]

my = mdf["label"]

### 학습용/평가용 분리
엔진 때와 똑같이 분리 — stratify로 정상/이상 비율 유지


In [ ]:
# 코드
mX_train, mX_test, my_train, my_test = train_test_split(mX, my, test_size=0.2, random_state=42, stratify=my)

### 학습과 예측
랜덤 포레스트로 학습하고 예측 — 코드 흐름이 엔진 때와 거의 동일


In [ ]:
# 코드
mmodel = RandomForestClassifier(n_estimators=100, random_state=42)
# mmodel

mmodel.fit(mX_train, my_train)
my_pred =mmodel.predict(mX_test)
my_pred[:10]

## 소리 분류 정리
- 데이터만 바뀌었을 뿐 흐름은 동일 — 분류의 본질은 같음
## 종합 미니 실습 (전체 파이프라인)
- 불러오기 → 라벨 → X/y → 분리 → 학습 → 예측 전 과정을 한 흐름으로


### 불러오기와 라벨
데이터를 불러오고 RUL로 failure_soon 라벨 만들기

In [ ]:
# 코드
df = pd.read_csv("27_cmapss_fd001_sample.csv")
# df.head()

df["failure_soon"] = (df["RUL"] <= 30).astype(int)

# print(df["failure_soon"].value_counts())
# print(df["failure_soon"].value_counts(normalize=True))

### X/y 구성
센서 여섯 개를 X로, failure_soon을 y로 — RUL이 X에 없는지 확인


In [ ]:
# 코드
features = ["sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_11", "sensor_15"]

X = df[feats]
y = df["failure_soon"]

print(X.shape)
print(y.shape)

### 데이터 분리
학습용과 평가용으로 나누기 — stratify=y로 비율 유지


In [ ]:
# 코드
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 학습과 예측
랜덤 포레스트로 학습하고 예측 — 여섯 단계를 막힘 없이 이었나?


In [ ]:
# 코드
model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
# print("전수 여부 : ", (y_pred == y_test.values.all()) )

# compare = pd.DataFrame({"예측": y_pred, "실제": y_test.values})

# compare.head()

## 종합 완성 점검
- 여섯 단계를 스스로 완성 — 오류가 나면 어느 단계인지 찾아 고치기
## 종합 결과 읽고 정리
- 예측·실제·확신도·중요 센서를 정비 우선순위로 해석하며 마무리


### 결과 표와 눈비교
예측·실제·곧고장확률을 모아 앞부분 눈비교 — 점수는 다음 파트


In [ ]:
# 코드
proba = model.predict_proba(X_test)
result = pd.DataFrame({'예측': y_pred, '실제': y_test.values, '곧고장확률': proba[:, 1]})
result.head(15)

### 우선순위와 중요 센서
곧 고장 확률 순 우선 점검 목록 + 어떤 센서가 크게 쓰였는지 확인


In [ ]:
# 코드
result.sort_values('곧고장확률', ascending=False).head(10)
pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

## 한 문장으로 정리
곧 고장 확률 높은 설비부터 점검, 중요 센서 부위 우선 살피기, 사람이 검토

## 제조 AX를 하려면 어떤 데이터를 준비해야 하는가?
제조 AX의 출발점은 “AI 모델을 뭘 쓸까?”가 아니라 “어떤 현장 문제를 어떤 데이터로 판단하게 만들 것인가?”이다.

제조업에서는 이미 MES, PLC, 센서, 검사장비, ERP 등에 데이터가 많이 쌓이고 있습니다. 그런데 데이터가 많다고 AI를 바로 적용할 수 있는 것은 아닙니다.

중요한 것은 데이터를 서로 맥락 있게 연결하는 것입니다.

https://github.com/cdjung515sj-coder/KNA-Data-analysis-1st/blob/main/2.%20practice/06_Z-score/manufacturing_AX_summary.md